In [4]:
import os
CHROMA_PATH = os.path.join(os.getcwd(), "rag", "policy_vector_db")
print(f"ChromaDB Path: {CHROMA_PATH}")

ChromaDB Path: d:\RAG Chatbot Project\proof-of-concepts\rag\policy_vector_db


In [ ]:
import umap
import chromadb
import numpy as np
import textwrap
import plotly.graph_objects as go
import ipywidgets as widgets
from sentence_transformers import SentenceTransformer
from IPython.display import display, clear_output

# 1. RE-INITIALIZE VARIABLES (Independent of previous cells)
# CHROMA_PATH = "/content/drive/MyDrive/policy_vector_db"
COLLECTION_NAME = "company_policies"
MODEL_NAME = 'all-MiniLM-L6-v2'

# Load Embedding Model
model = SentenceTransformer(MODEL_NAME)

# Connect to ChromaDB
client = chromadb.PersistentClient(path=CHROMA_PATH)
collection = client.get_collection(name=COLLECTION_NAME)

# 2. FETCH DATA
all_data = collection.get(include=["embeddings", "metadatas", "documents"])
all_embeddings = np.array(all_data['embeddings'])
all_docs = all_data['documents']
all_sources = [meta['source'] for meta in all_data['metadatas']]
all_ids = all_data['ids']

# 3. COMPUTE 2D PROJECTION (The Galaxy)
print("Computing 2D projection... (Stand-alone mode)")
reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, metric='cosine', random_state=42)
projections = reducer.fit_transform(all_embeddings)

# 4. UI SETUP & RENDERING
query_input = widgets.Text(placeholder='Type query to highlight nodes...', description='Search:', layout={'width': '80%'})
search_button = widgets.Button(description='Highlight Results', button_style='info')
out = widgets.Output()

def render_graph(highlight_ids=[]):
    with out:
        clear_output(wait=True)
        
        # Color logic: Red for matches, semi-transparent gray for others
        colors = ['red' if tid in highlight_ids else 'rgba(150, 150, 150, 0.4)' for tid in all_ids]
        sizes = [12 if tid in highlight_ids else 6 for tid in all_ids]
        
        fig = go.Figure()
        
        # Add all policy chunks as the background galaxy
        fig.add_trace(go.Scatter(
            x=projections[:, 0], 
            y=projections[:, 1],
            mode='markers',
            marker=dict(color=colors, size=sizes, line=dict(width=0.5, color='white')),
            text=[f"<b>{src}</b><br>{textwrap.shorten(doc, width=100)}" for src, doc in zip(all_sources, all_docs)],
            hoverinfo='text'
        ))

        fig.update_layout(
            title="2D Vector Space Visualization (Independent Cell)",
            template="plotly_dark",
            height=600,
            margin=dict(l=0, r=0, b=0, t=40),
            showlegend=False
        )
        # Use 'colab' renderer for guaranteed display in Google Colab
        fig.show(renderer="colab")

def on_click(b):
    if not query_input.value: return
    # Use the local model variable to encode the query
    query_emb = model.encode(query_input.value).tolist()
    results = collection.query(query_embeddings=[query_emb], n_results=5)
    render_graph(highlight_ids=results['ids'][0])

search_button.on_click(on_click)

# Initial execution
display(widgets.VBox([widgets.HBox([query_input, search_button]), out]))
render_graph()

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Computing 2D projection... (Stand-alone mode)


d:\RAG Chatbot Project\.venv\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
